# 03 – HuggingFace Proxy Demo

Compare **direct** HuggingFace Hub access vs **Artifactory proxy**. When you use the proxy, all model downloads go through Artifactory: they are cached, scanned by Xray, and governed by Curation policies.

In [ ]:
# Load .env if present
import os
from pathlib import Path

try:
    env_file = Path(__file__).parent / ".env"
except NameError:
    env_file = Path(".env")
if not env_file.exists():
    env_file = Path("../.env")
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip("'\""))

hf_endpoint = os.environ.get("HF_ENDPOINT")
hf_token = os.environ.get("HF_TOKEN")
print(f"HF_ENDPOINT: {hf_endpoint or '(not set)'}")
print(f"HF_TOKEN: {'***' if hf_token else '(not set)'}")

## How the proxy works

| Mode | HF_ENDPOINT | HF_TOKEN | Source |
|------|-------------|----------|--------|
| **Direct** | not set | not set | HuggingFace Hub (public) |
| **Proxy** | `https://.../artifactory/api/huggingfaceml/hf-remote` | Artifactory token | Artifactory → cache → HuggingFace |

With the proxy, models are cached in Artifactory and can be scanned by Xray before use.

In [ ]:
from huggingface_hub import HfApi

model_id = "distilbert-base-uncased-finetuned-sst-2-english"
api = HfApi()

# Show model info (works with or without proxy)
info = api.model_info(model_id)
print(f"Model: {info.modelId}")
print(f"Downloads: {info.downloads}")
print(f"Tags: {info.tags[:5] if info.tags else 'N/A'}...")

## Load model (uses proxy if HF_ENDPOINT is set)

In [ ]:
from transformers import AutoTokenizer

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

if hf_endpoint:
    print(f"\nUsing Artifactory proxy: {hf_endpoint}")
    print("Model was downloaded through Artifactory (cached, scannable).")
else:
    print("\nUsing direct HuggingFace Hub (no proxy).")
    print("Set HF_ENDPOINT and HF_TOKEN to use the Artifactory proxy.")

print("\nTokenizer loaded successfully.")

## Quick inference (optional)

In [ ]:
from transformers import AutoModelForSequenceClassification
import torch

model = AutoModelForSequenceClassification.from_pretrained(model_id)
text = "This demo is great!"
inputs = tokenizer(text, return_tensors="pt")
model.eval()
with torch.no_grad():
    out = model(**inputs)
probs = torch.nn.functional.softmax(out.logits, dim=-1)
label_id = probs.argmax().item()
score = probs.max().item()
label = model.config.id2label[label_id]
print(f"{text!r} -> {label} ({score:.3f})")